# Metronome Compass Testing

This notebook provides utilities for testing the complex `metronome_compass` (full battle tracking) by generating seeds that produce specific outcomes.

In [6]:
%load_ext autoreload
%autoreload 2
from claytonlib.safari import advance_rng
from claytonlib.metronome_compass import precompute_path, render_path
from claytonlib.moves import _moves_by_number, resolve_move
import datetime as dt

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## RNG Reversing Utility

LCRNG: $state_{n+1} = (state_n \cdot 1103515245 + 24691) \pmod{2^{32}}$

Inverse: $state_n = ((state_{n+1} - 24691) \cdot 0xEEB9EB65) \pmod{2^{32}}$

In [7]:
def reverse_rng(state: int, n: int = 1) -> int:
    """Backtrack the LCRNG n steps."""
    inv_mult = 0xEEB9EB65
    for _ in range(n):
        state = ((state - 24691) * inv_mult) & 0xFFFFFFFF
    return state


def generate_seed_for_move(
    move_name: str,
    magikarp_level: int = 2,
    n_turns: int = 3,
    verify_starting_turn: bool = False,
    find_x: int | None = None,
    criteria_substrings: list[str] | None = None,
    max_attempts: int = 1000,
):
    """Find seeds producing a specific Metronome move on Turn 1.

    Turn order: Magikarp moves first, then Metronome user.
    Advances before the Metronome roll (advance #N → seed = reverse_rng(target, N)):
      Splash (any level):  6+1+4+0+2+2 = 15  → N=16  (check rev[9]  even  → Splash)
      Tackle miss:         6+1+4+3+0+2 = 16  → N=17  (check rev[10] odd + rev[3]  >=95 → miss)
      Tackle hit:          6+1+4+3+2+2 = 18  → N=19  (check rev[12] odd + rev[5]  <95  → hit)

    Returns:
      verify_starting_turn=True  → dict mapping scenario → seed
      find_x set                 → list of up to find_x matching seeds (possibly empty)
      default                    → single matching seed (int) or None
    """
    move = resolve_move(move_name)
    if not move or not move.metronome_usable:
        raise ValueError(f"Move {move_name!r} is not metronome-usable.")

    target_num = move.number
    pool = 467
    target_val = target_num - 1

    def _check_criteria(seed: int) -> bool:
        if not criteria_substrings:
            return True
        path = precompute_path(seed, magikarp_level=magikarp_level, opposite_gender=False, n_turns=n_turns)
        rendered = render_path(path)
        return all(s in rendered for s in criteria_substrings)

    def _rev19(attempt: int):
        top16 = (target_val + pool * attempt) % 65536
        target_state = top16 << 16
        rev = [None]
        s = target_state
        for _ in range(19):
            s = ((s - 24691) * 0xEEB9EB65) & 0xFFFFFFFF
            rev.append(s)
        return rev

    if verify_starting_turn:
        results = {}
        target_count = 1 if magikarp_level < 15 else 3
        for attempt in range(max_attempts):
            rev = _rev19(attempt)
            if magikarp_level < 15:
                if 'splash' not in results and _check_criteria(rev[16]):
                    results['splash'] = rev[16]
            else:
                if 'splash' not in results and (rev[9] >> 16) % 2 == 0 and _check_criteria(rev[16]):
                    results['splash'] = rev[16]
                if 'tackle_hit' not in results and (rev[12] >> 16) % 2 == 1 and (rev[5] >> 16) % 100 < 95 and _check_criteria(rev[19]):
                    results['tackle_hit'] = rev[19]
                if 'tackle_miss' not in results and (rev[10] >> 16) % 2 == 1 and (rev[3] >> 16) % 100 >= 95 and _check_criteria(rev[17]):
                    results['tackle_miss'] = rev[17]
            if len(results) == target_count:
                break
        return results

    collected = []
    for attempt in range(max_attempts):
        rev = _rev19(attempt)
        candidates = []
        if magikarp_level < 15:
            candidates.append(rev[16])
        else:
            if (rev[9] >> 16) % 2 == 0:
                candidates.append(rev[16])
            if (rev[12] >> 16) % 2 == 1 and (rev[5] >> 16) % 100 < 95:
                candidates.append(rev[19])
            if (rev[10] >> 16) % 2 == 1 and (rev[3] >> 16) % 100 >= 95:
                candidates.append(rev[17])

        for seed in candidates:
            if _check_criteria(seed):
                if find_x is not None:
                    collected.append(seed)
                    if len(collected) >= find_x:
                        return collected
                else:
                    return seed

    return collected if find_x is not None else None


def verify_seed(seed: int, magikarp_level: int = 2, n_turns: int = 3):
    path = precompute_path(seed, magikarp_level=magikarp_level, opposite_gender=False, n_turns=n_turns)
    print(f"Seed: 0x{seed:08X}")
    print(f"Path: {render_path(path)}")
    moves = _moves_by_number()
    if path:
        for token in path[0]:
            if hasattr(token, 'move_num'):
                print(f"Move: {moves[token.move_num].name} (M{token.move_num:03d})")
                break


## Example: Seed for Flamethrower

In [ ]:
seed = generate_seed_for_move("Flamethrower")
verify_seed(seed)


## Example: Seed for Splash (Turn 1 Metronome)

In [ ]:
seed = generate_seed_for_move("Splash")
verify_seed(seed)


## Testing other moves

In [ ]:
seed = generate_seed_for_move("U-turn")
verify_seed(seed)


## Multi-Turn Path Generation

In [13]:
seed = 0x0a7d8651
path = precompute_path(seed, magikarp_level=15, opposite_gender=False, n_turns=5)
print(f"Seed: 0x{seed:08X} (Level 15 Magikarp)")
print(f"Path: {render_path(path)}")

Seed: 0x0A7D8651 (Level 15 Magikarp)
Path: KspM109h KtkCFZM316 KspM285 Ktk-M040h KtkCFZSCFZM130


## All 3 Magikarp Scenarios (Level 15)

In [ ]:
seeds = generate_seed_for_move("False Swipe", magikarp_level=15, verify_starting_turn=True)
moves = _moves_by_number()

for scenario, seed in seeds.items():
    print(f"=== {scenario} ===")
    verify_seed(seed, magikarp_level=15)
    print()


In [12]:
seed = 0x0a7d8651
for _ in range(30):
    val = seed >> 16
    print(f"Seed: 0x{seed:08X} - Val {val}")
    seed = advance_rng(seed)

Seed: 0xA30D0469 - Val 41741
Seed: 0x76193F28 - Val 30233
Seed: 0xFAEE747B - Val 64238
Seed: 0xD12772D2 - Val 53543
Seed: 0x33343FDD - Val 13108
Seed: 0x3AA2E78C - Val 15010
Seed: 0x3C319F0F - Val 15409
Seed: 0x9431ABD6 - Val 37937
Seed: 0x7905BE91 - Val 30981
Seed: 0x7CA8B230 - Val 31912
Seed: 0x7B3EDEE3 - Val 31550
Seed: 0xA840711A - Val 43072
Seed: 0x5A027485 - Val 23042
Seed: 0xF66A8314 - Val 63082
Seed: 0xD4C247F7 - Val 54466
Seed: 0x28B0469E - Val 10416
Seed: 0xC4C695B9 - Val 50374
Seed: 0x10427E38 - Val 4162
Seed: 0x06152E4B - Val 1557
Seed: 0xE421F062 - Val 58401
Seed: 0x477D962D - Val 18301
Seed: 0x4809079C - Val 18441
Seed: 0xEAD225DF - Val 60114
Seed: 0xEC7E7266 - Val 60542
Seed: 0x999629E1 - Val 39318
Seed: 0xAAB8C340 - Val 43704
Seed: 0x3FA902B3 - Val 16297
Seed: 0x233B10AA - Val 9019
Seed: 0x0CB644D5 - Val 3254
Seed: 0x44529524 - Val 17490


In [8]:
## One seed per metronome-usable move (level 15 Magikarp, 6 turns)
import json as _json
import re as _re

with open('claytonlib/basedata/moves.json') as _f:
    _all_moves_data = _json.load(_f)

_metronome_moves = [m for m in _all_moves_data if m['metronome_usable']]
_moves_lookup = _moves_by_number()
print(f"Generating seeds for {len(_metronome_moves)} metronome-usable moves (level 15 Magikarp, 6 turns)...\n")

_seed_map = {}
_aborted = False
for _move_data in _metronome_moves:
    _name = _move_data['name']
    _expected_num = _move_data['number']
    _seed = generate_seed_for_move(_name, magikarp_level=15, n_turns=6)
    if _seed is None:
        print(f"  WARNING: no seed found for {_name}")
        continue
    _path = precompute_path(_seed, magikarp_level=15, opposite_gender=False, n_turns=6)
    _path_str = render_path(_path)
    _move_nums = [int(m) for m in _re.findall(r'M(\d{3})', _path_str)]
    _move_names = [_moves_lookup[n].name for n in _move_nums if n in _moves_lookup]
    if not _move_nums or _move_nums[0] != _expected_num:
        _found = _moves_lookup[_move_nums[0]].name if _move_nums else 'none'
        print(f"ERROR: {_name} — expected M{_expected_num:03d} first but got {_found}. Aborting.")
        _aborted = True
        break
    _seed_map[f"0x{_seed:08X}"] = {
        "verification": 0,
        "path": _path_str,
        "moves": _move_names,
    }

if not _aborted:
    print(f"Built map with {len(_seed_map)} entries.\n")
    print(_json.dumps(_seed_map, indent=2))


Generating seeds for 442 metronome-usable moves (level 15 Magikarp, 6 turns)...

Built map with 442 entries.

{
  "0x8552B0CF": {
    "verification": 0,
    "path": "KtkhM001! KspM344h KtkhM186- KspM031hhhh! KtkhM311h KspM216h",
    "moves": [
      "Pound",
      "Volt Tackle",
      "Sweet Kiss",
      "Fury Attack",
      "Weather Ball",
      "Return"
    ]
  },
  "0xB2943EF0": {
    "verification": 0,
    "path": "KspM002h KtkhM137- KspM437h~ KtkhM377h KtkhM236 KtkhM085h",
    "moves": [
      "Karate Chop",
      "Glare",
      "Leaf Storm",
      "Heal Block",
      "Moonlight",
      "Thunderbolt"
    ]
  },
  "0x860CB0CF": {
    "verification": 0,
    "path": "KtkhM003hh KtkhM282h Ktk-M241 KspM315h~ KtkhM175h KspM467",
    "moves": [
      "Double Slap",
      "Knock Off",
      "Sunny Day",
      "Overheat",
      "Flail",
      "Shadow Force"
    ]
  },
  "0xCE163EF0": {
    "verification": 0,
    "path": "KspM004!hh KspM057h KtkhM429h~ KspM148h KtkhM006h KspM445",
    "move